In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torch
import os, sys

# 自动定位 model/ 目录（含 mrmpformer/）加入 sys.path，无论从哪启动 jupyter
_nb_dir = os.getcwd()
for _c in [_nb_dir, os.path.dirname(_nb_dir), os.path.dirname(os.path.dirname(_nb_dir))]:
    if os.path.isdir(os.path.join(_c, "mrmpformer")):
        if _c not in sys.path:
            sys.path.insert(0, _c)
        break

from mrmpformer.hubconf import *
from mrmpformer.util.misc import nested_tensor_from_tensor_list

# 解决 Windows 下 PyTorch/numpy 的 OpenMP 运行时冲突
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

torch.set_grad_enabled(False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CLASSES = ['peak']

# 图像预处理：与 predict_utils 一致（ToTensor + Normalize，不 Resize）
transform = T.Compose([
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


def _load_model(model_path):
    model = quan_former()
    state_dict = torch.load(model_path, map_location='cpu', weights_only=False)
    model.to(device)
    model.load_state_dict(state_dict["model"])
    model.eval()
    return model


def _feat_map_shape(feat_output):
    """从 backbone hook 输出提取特征图张量与空间尺寸 (H, W)。"""
    feat = feat_output
    # Joiner 返回 (features, pos)；features 可能是 NestedTensor 或 tuple
    if isinstance(feat, (list, tuple)):
        feat = feat[0]
    # DETR backbone 返回 OrderedDict {"0": NestedTensor}
    if isinstance(feat, dict):
        feat = feat.get("0", next(iter(feat.values())))
    tensors = getattr(feat, "tensors", None)
    if tensors is None:
        if isinstance(feat, (list, tuple)):
            feat = feat[0]
        tensors = getattr(feat, "tensors", feat)
    if isinstance(tensors, (list, tuple)):
        tensors = tensors[0]
    # tensors: [B, C, H, W]
    return tensors, tensors.shape[-2], tensors.shape[-1]


def _run_with_hooks(model, img):
    conv_features, enc_attn_weights, dec_attn_weights = [], [], []
    hooks = [
        model.backbone[-2].register_forward_hook(
            lambda self, input, output: conv_features.append(output)
        ),
        model.transformer.encoder.layers[-1].self_attn.register_forward_hook(
            lambda self, input, output: enc_attn_weights.append(output[1])
        ),
        model.transformer.decoder.layers[-1].multihead_attn.register_forward_hook(
            lambda self, input, output: dec_attn_weights.append(output[1])
        ),
    ]
    outputs = model(img)
    for h in hooks:
        h.remove()
    return outputs, conv_features, enc_attn_weights, dec_attn_weights


def visualize_decoder(model_path, img_path):
    """可视化 decoder 交叉注意力：每个 query 关注图像哪些空间区域。"""
    model = _load_model(model_path)
    im = Image.open(img_path).convert('RGB')
    img = transform(im).to(device)
    img = nested_tensor_from_tensor_list([img])

    outputs, conv_features, enc_attn_weights, dec_attn_weights = _run_with_hooks(model, img)

    # dec_attn_weights[0]: [batch, num_queries, num_pixels]（可能含 n_heads 维）
    dec_attn = dec_attn_weights[0]
    if dec_attn.dim() == 4:  # [batch, n_heads, queries, pixels]
        dec_attn = dec_attn.mean(dim=1)
    if dec_attn.dim() == 3:  # [batch, queries, pixels]
        dec_attn = dec_attn[0]  # 去 batch -> [queries, pixels]
    _, h_feat, w_feat = _feat_map_shape(conv_features[0])

    n_queries = dec_attn.shape[0]
    fig, axs = plt.subplots(1, n_queries + 1, figsize=(4 * (n_queries + 1), 4))
    if not isinstance(axs, list):
        axs = [axs] if n_queries == 0 else list(axs)
    axs[0].imshow(im)
    axs[0].set_title("input")
    axs[0].axis("off")
    for qi in range(n_queries):
        attn_map = dec_attn[qi].reshape(h_feat, w_feat).cpu().detach()
        axs[qi + 1].imshow(attn_map, cmap='viridis', aspect='auto')
        axs[qi + 1].set_title(f"query {qi}")
        axs[qi + 1].axis("off")
    plt.tight_layout()
    plt.show()
    return outputs


def visualize_encoder(model_path, img_path):
    """可视化 encoder 自注意力（平均各 head，展示注意力矩阵热力图）。"""
    model = _load_model(model_path)
    im = Image.open(img_path).convert('RGB')
    img = transform(im).to(device)
    img = nested_tensor_from_tensor_list([img])

    outputs, conv_features, enc_attn_weights, dec_attn_weights = _run_with_hooks(model, img)

    # enc_attn_weights[0]: [batch, n_heads, pixels, pixels] 或 [batch, pixels, pixels]
    enc_attn = enc_attn_weights[0]
    if enc_attn.dim() == 4:
        avg_attn = enc_attn.mean(dim=(0, 1))
    elif enc_attn.dim() == 3:
        avg_attn = enc_attn.mean(dim=0)
    else:
        avg_attn = enc_attn
    _, h_feat, w_feat = _feat_map_shape(conv_features[0])
    n_pix = h_feat * w_feat

    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    axs[0].imshow(im)
    axs[0].set_title("input")
    axs[0].axis("off")
    # 取前 n_pix 行列的注意力子矩阵
    sub = avg_attn[:n_pix, :n_pix].cpu().detach()
    im_attn = axs[1].imshow(sub.numpy(), cmap='viridis', aspect='auto')
    axs[1].set_title("encoder self-attn (avg heads)")
    plt.colorbar(im_attn, ax=axs[1], fraction=0.046)
    # 特征图（第一通道）
    feat_tensors, _, _ = _feat_map_shape(conv_features[0])
    feat_ch0 = feat_tensors[0, 0].cpu().detach()
    axs[2].imshow(feat_ch0, cmap='viridis', aspect='auto')
    axs[2].set_title("feature map (ch0)")
    axs[2].axis("off")
    plt.tight_layout()
    plt.show()
    return outputs


In [ ]:
images_path = '../data/test/xic_test/20260715_食药院_测试_1/2_mz895.5000_q3751.5000_阿维菌素-1.jpeg'
model_path = 'checkpoint/checkpoint0029.pth'
visualize_encoder(model_path, images_path)

In [ ]:
visualize_decoder(model_path, images_path)

In [ ]:
images_path = '../data/test/xic_test/20260715_食药院_测试_1/3_mz895.5000_q3449.0000_阿维菌素-2.jpeg'
visualize_encoder(model_path, images_path)